### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

In [36]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory): # defines a function to process all PDFs in a directory and its parameter can be called anything
    """Process all PDF files in the specified directory."""
    all_documents = [] # Initialize an empty list to store all document objects (pages returned by the loader)
    pdf_dir = Path(pdf_directory) 
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.rglob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files.")
    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file}")
        try:
            # Load the PDF file
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f" loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
    # Process all PDFs in the data directory

all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files.
Processing file: ../data/pdf/Owais's Resume (2).pdf
 loaded 1 pages
Processing file: ../data/pdf/sample-local-pdf.pdf
 loaded 3 pages

Total documents loaded: 4


In [38]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': "../data/pdf/Owais's Resume (2).pdf", 'file_path': "../data/pdf/Owais's Resume (2).pdf", 'total_pages': 1, 'format': 'PDF 1.4', 'title': "Owais's Resume", 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': "Owais's Resume (2).pdf", 'file_type': 'pdf'}, page_content='Owais Ahmed \nChicago, IL| https://www.linkedin.com/in/oahme3 | 224-518-5004 | owaisahmed136@gmail.com \n \n \nEXPERIENCE \nThe Disrupt Labs\u200b\n Remote \nLLM Intern\u200b\nOct 2025-Present \n●\u200b Built and deployed AI-driven coaching systems leveraging Retrieval-Augmented Generation (RAG) \narchitecture with LangChain for dynamic context retrieval. \n●\u200b Integrated Pinecone vector databases to enable efficient semantic search and improve the relevance of \nmodel-generated responses. \n●\u200b Implemented Groq-based i

In [37]:
### Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...") # Print first 200 characters
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs